# Treinamento da EfficientNetB0 com PyTorch e GPU

Notebook adaptado para treinar a EfficientNetB0 usando PyTorch/torchvision com CUDA no Windows. Mantém a divisão 60/20/20, aumento de dados no treino, pesos por classe, treinamento da cabeça do modelo e fine-tuning.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sem GPU")

In [ ]:
def find_project_dir() -> Path:
    """Encontra a raiz do projeto a partir da pasta atual."""
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)

    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "Não foi possível encontrar a raiz do projeto. "
        "Abra o notebook dentro da pasta do projeto ou ajuste PROJECT_DIR manualmente."
    )


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
SPLITS_DIR = DATA_DIR / "splits"
RAW_IMAGES_DIR = DATA_DIR / "raw" / "train_images"

RESULTS_DIR = PROJECT_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [METRICS_DIR, FIGURES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_IMAGES_DIR:", RAW_IMAGES_DIR)
print("SPLITS_DIR:", SPLITS_DIR)

In [ ]:
MODEL_NAME = "EfficientNetB0"
MODEL_KEY = "efficientnetb0_pytorch_gpu"
EXPERIMENT_NAME = "v1_pytorch_gpu_ft_features_lr1e5"
MODEL_OUTPUT_KEY = f"{MODEL_KEY}_{EXPERIMENT_NAME}"

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
NUM_CLASSES = 5
SEED = 42

EPOCHS_HEAD = 10
EPOCHS_FINE_TUNING = 20

HEAD_LEARNING_RATE = 1e-3
FINE_TUNING_LEARNING_RATE = 1e-5
DROPOUT_RATE = 0.3
CLASS_WEIGHT_MODE = "balanced"  # opções: balanced, sqrt, none

NUM_WORKERS = 0  # no Windows, 0 é mais estável para notebooks
PIN_MEMORY = torch.cuda.is_available()

MODEL_OUTPUT_DIR = MODELS_DIR / MODEL_OUTPUT_KEY
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Modelo:", MODEL_NAME)
print("Experimento:", EXPERIMENT_NAME)
print("Batch size:", BATCH_SIZE)

## Carregamento dos splits

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_split.csv")


def rebuild_image_paths(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["image_path"] = df["id_code"].apply(
        lambda image_id: str((RAW_IMAGES_DIR / f"{image_id}.png").resolve())
    )
    return df


train_df = rebuild_image_paths(train_df)
val_df = rebuild_image_paths(val_df)
test_df = rebuild_image_paths(test_df)

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)
train_df.head()

In [ ]:
class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa",
}


def verify_images(df: pd.DataFrame, name: str) -> None:
    missing = df[~df["image_path"].apply(lambda path: Path(path).exists())]
    print(f"{name}: total={len(df)} | ausentes={len(missing)}")
    if len(missing) > 0:
        display(missing.head())
        raise FileNotFoundError(f"Existem imagens ausentes em {name}.")


verify_images(train_df, "Treino")
verify_images(val_df, "Validação")
verify_images(test_df, "Teste")

## Pesos por classe

In [ ]:
classes = np.unique(train_df["diagnosis"].values)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values,
)

class_weights_balanced = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, class_weights_values)
}

class_weights_sqrt = {
    class_id: float(np.sqrt(weight))
    for class_id, weight in class_weights_balanced.items()
}

if CLASS_WEIGHT_MODE == "balanced":
    class_weights = class_weights_balanced
elif CLASS_WEIGHT_MODE == "sqrt":
    class_weights = class_weights_sqrt
elif CLASS_WEIGHT_MODE == "none":
    class_weights = None
else:
    raise ValueError("CLASS_WEIGHT_MODE deve ser 'balanced', 'sqrt' ou 'none'.")

print("Class weights balanceados:")
print(class_weights_balanced)

print("\nClass weights suavizados:")
print(class_weights_sqrt)

print("\nClass weights usados neste experimento:")
print(class_weights)

## Dataset e DataLoader

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

val_test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


class AptosDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["image_path"]
        label = int(row["diagnosis"])

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = AptosDataset(train_df, transform=train_transform)
val_dataset = AptosDataset(val_df, transform=val_test_transform)
test_dataset = AptosDataset(test_df, transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

images, labels = next(iter(train_loader))
print("Batch de imagens:", images.shape)
print("Batch de rótulos:", labels.shape)
print("Dtype imagens:", images.dtype)
print("Dtype labels:", labels.dtype)

## Construção do modelo

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Congela a base convolucional para a primeira fase.
for param in model.parameters():
    param.requires_grad = False

# Substitui a cabeça final para 5 classes.
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(device)
print(model.classifier)

# Teste rápido de passagem pelo modelo.
with torch.no_grad():
    sample_images = images.to(device)
    sample_outputs = model(sample_images)
    print("Saída do modelo:", sample_outputs.shape)

## Funções de treinamento e avaliação

In [ ]:
def get_current_lr(optimizer):
    return optimizer.param_groups[0]["lr"]


def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)
            preds = torch.argmax(outputs, dim=1)

            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = batch_images.size(0)
        running_loss += loss.item() * batch_size
        running_corrects += torch.sum(preds == batch_labels).item()
        total_samples += batch_size

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples

    return epoch_loss, epoch_acc


def fit_phase(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs,
    checkpoint_path,
    phase_name,
    patience=6,
):
    history = []
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        val_loss, val_acc = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
            device=device,
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        lr = get_current_lr(optimizer)

        row = {
            "phase": phase_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": lr,
        }
        history.append(row)

        print(
            f"[{phase_name}] Época {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={lr:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  Melhor modelo salvo em: {checkpoint_path}")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"  Early stopping na época {epoch}. Melhor época: {best_epoch}.")
            break

    elapsed_time = time.time() - start_time

    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    return history, elapsed_time


def predict_loader(model, loader, device):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(device, non_blocking=True)
            outputs = model(batch_images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            y_pred.extend(preds.tolist())
            y_true.extend(batch_labels.numpy().tolist())

    return np.array(y_true), np.array(y_pred)

## Treinamento da cabeça

In [ ]:
if class_weights is not None:
    class_weight_tensor = torch.tensor(
        [class_weights[i] for i in range(NUM_CLASSES)],
        dtype=torch.float32,
        device=device,
    )
else:
    class_weight_tensor = None

criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)

optimizer_head = optim.Adam(
    model.classifier.parameters(),
    lr=HEAD_LEARNING_RATE,
)

scheduler_head = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_head,
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
)

checkpoint_head_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_head_best.pt"

history_head, head_training_time = fit_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_head,
    scheduler=scheduler_head,
    device=device,
    epochs=EPOCHS_HEAD,
    checkpoint_path=checkpoint_head_path,
    phase_name="head",
)

print(f"Tempo de treinamento da cabeça: {head_training_time:.2f} segundos")

## Fine-tuning

In [ ]:
# Carrega o melhor checkpoint da cabeça antes do fine-tuning.
if checkpoint_head_path.exists():
    model.load_state_dict(torch.load(checkpoint_head_path, map_location=device))

# Congela tudo novamente.
for param in model.parameters():
    param.requires_grad = False

# Mantém a cabeça treinável.
for param in model.classifier.parameters():
    param.requires_grad = True

# Descongela os últimos blocos da EfficientNetB0.
# A lista model.features contém os blocos convolucionais da EfficientNet.
for block in model.features[-2:]:
    for param in block.parameters():
        param.requires_grad = True

trainable_params = [param for param in model.parameters() if param.requires_grad]
print("Parâmetros treináveis:", sum(param.numel() for param in trainable_params))

optimizer_fine = optim.Adam(
    trainable_params,
    lr=FINE_TUNING_LEARNING_RATE,
)

scheduler_fine = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fine,
    mode="min",
    factor=0.2,
    patience=3,
    min_lr=1e-7,
)

checkpoint_fine_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_fine_best.pt"

history_fine, fine_tuning_time = fit_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_fine,
    scheduler=scheduler_fine,
    device=device,
    epochs=EPOCHS_FINE_TUNING,
    checkpoint_path=checkpoint_fine_path,
    phase_name="fine_tuning",
)

print(f"Tempo de fine-tuning: {fine_tuning_time:.2f} segundos")

## Histórico de treinamento

In [ ]:
history_df = pd.DataFrame(history_head + history_fine)
history_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_history.csv"
history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

print("Histórico salvo em:", history_path)
history_df.tail()

In [ ]:
plt.figure(figsize=(8, 5))
for phase in history_df["phase"].unique():
    phase_df = history_df[history_df["phase"] == phase]
    plt.plot(phase_df.index + 1, phase_df["train_loss"], label=f"Train loss - {phase}")
    plt.plot(phase_df.index + 1, phase_df["val_loss"], label=f"Val loss - {phase}")

plt.title(f"Loss durante o treinamento - {MODEL_NAME} PyTorch")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_loss.png"
plt.savefig(loss_fig_path, dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
for phase in history_df["phase"].unique():
    phase_df = history_df[history_df["phase"] == phase]
    plt.plot(phase_df.index + 1, phase_df["train_accuracy"], label=f"Train acc - {phase}")
    plt.plot(phase_df.index + 1, phase_df["val_accuracy"], label=f"Val acc - {phase}")

plt.title(f"Acurácia durante o treinamento - {MODEL_NAME} PyTorch")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.tight_layout()

acc_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_accuracy.png"
plt.savefig(acc_fig_path, dpi=300)
plt.show()

print("Figuras salvas:")
print(loss_fig_path)
print(acc_fig_path)

## Avaliação no conjunto de teste

In [ ]:
# Carrega o melhor checkpoint do fine-tuning, se existir.
if checkpoint_fine_path.exists():
    model.load_state_dict(torch.load(checkpoint_fine_path, map_location=device))
elif checkpoint_head_path.exists():
    model.load_state_dict(torch.load(checkpoint_head_path, map_location=device))

test_loss, test_accuracy = run_epoch(
    model=model,
    loader=test_loader,
    criterion=criterion,
    optimizer=None,
    device=device,
)

print(f"Loss no teste: {test_loss:.4f}")
print(f"Acurácia no teste: {test_accuracy:.4f}")

In [ ]:
y_true, y_pred = predict_loader(model, test_loader, device)

report = classification_report(
    y_true,
    y_pred,
    target_names=[class_names[i] for i in range(NUM_CLASSES)],
    digits=4,
)

print(report)

report_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_classification_report.txt"
report_path.write_text(report, encoding="utf-8")

print("Relatório salvo em:", report_path)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[class_names[i] for i in range(NUM_CLASSES)],
    columns=[class_names[i] for i in range(NUM_CLASSES)],
)

cm_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.csv"
cm_df.to_csv(cm_path, encoding="utf-8-sig")

plt.figure(figsize=(9, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[class_names[i] for i in range(NUM_CLASSES)],
)
disp.plot(cmap="viridis", values_format="d", xticks_rotation=45)
plt.title(f"Matriz de Confusão - {MODEL_NAME} PyTorch")
plt.tight_layout()

cm_fig_path = FIGURES_DIR / f"{MODEL_OUTPUT_KEY}_confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Matriz CSV salva em:", cm_path)
print("Figura salva em:", cm_fig_path)

## Salvamento do modelo e resumo

In [ ]:
final_model_path = MODEL_OUTPUT_DIR / f"{MODEL_OUTPUT_KEY}_final.pt"
torch.save(model.state_dict(), final_model_path)

summary_metrics = {
    "model": MODEL_NAME,
    "framework": "PyTorch",
    "experiment_name": EXPERIMENT_NAME,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "epochs_head_configured": EPOCHS_HEAD,
    "epochs_fine_tuning_configured": EPOCHS_FINE_TUNING,
    "epochs_completed_total": int(len(history_df)),
    "batch_size": BATCH_SIZE,
    "image_size": list(IMG_SIZE),
    "dropout_rate": DROPOUT_RATE,
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weights": class_weights,
    "split_strategy": "60% treino / 20% validação / 20% teste",
    "preprocess_input": "torchvision.transforms.Normalize(mean=ImageNet, std=ImageNet)",
    "data_augmentation": {
        "Resize": list(IMG_SIZE),
        "RandomRotation": 8,
        "RandomAffine_translate": [0.03, 0.03],
        "RandomAffine_scale": [0.95, 1.05],
        "RandomHorizontalFlip": 0.5,
    },
    "run_fine_tuning": True,
    "fine_tuning_layers": "features[-2:] + classifier",
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tuning_learning_rate": FINE_TUNING_LEARNING_RATE,
    "head_training_time_seconds": float(head_training_time),
    "fine_tuning_time_seconds": float(fine_tuning_time),
    "gpu_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "final_model_path": str(final_model_path),
    "best_head_checkpoint_path": str(checkpoint_head_path),
    "best_fine_checkpoint_path": str(checkpoint_fine_path),
}

summary_path = METRICS_DIR / f"{MODEL_OUTPUT_KEY}_summary_metrics.json"
with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary_metrics, file, ensure_ascii=False, indent=4)

print("Modelo final salvo em:", final_model_path)
print("Resumo salvo em:", summary_path)
print(json.dumps(summary_metrics, ensure_ascii=False, indent=4))